In [9]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import time


device="cuda"

transform=v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32,scale=True)])
train_data=datasets.FashionMNIST(root="data",train=True,download=True,transform=transform)
test_data=datasets.FashionMNIST(root="data",train=False,download=False,transform=transform)
train_loader=DataLoader(train_data,batch_size=256,shuffle=True)
test_loader=DataLoader(test_data,batch_size=256,shuffle=False)

class NeuralNetwork(nn.Module):

    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1=nn.Linear(784,256)
        self.relu1=nn.ReLU()
        self.layer2=nn.Linear(256,128)
        self.relu2=nn.ReLU()
        self.dropout = nn.Dropout(p=0.3)

        self.layer3=nn.Linear(128,10)

    def forward(self,x):
        x=self.flatten(x)
        x=self.layer1(x)
        x=self.relu1(x)
        x=self.dropout(x)
        x=self.layer2(x)
        x=self.relu2(x)
        x=self.dropout(x)
        x=self.layer3(x)
        return x

model=NeuralNetwork().to(device)

loss=nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
for epoch in range(20):
    model.train()
    total_loss=0
    for x,y in train_loader:
        x=x.to(device)
        y=y.to(device)
        logits=model(x)
        loss1=loss(logits,y)
        optimizer.zero_grad()
        loss1.backward()
        optimizer.step()
        total_loss+=loss1.item()
    if epoch%2==0:
         average_loss = total_loss / len(train_loader)
         print(f"Epoch: {epoch} | "f"Loss: {average_loss:.4f}")

Epoch: 0 | Loss: 0.7252
Epoch: 2 | Loss: 0.4047
Epoch: 4 | Loss: 0.3574
Epoch: 6 | Loss: 0.3332
Epoch: 8 | Loss: 0.3135
Epoch: 10 | Loss: 0.3013
Epoch: 12 | Loss: 0.2871
Epoch: 14 | Loss: 0.2747
Epoch: 16 | Loss: 0.2694
Epoch: 18 | Loss: 0.2597


In [10]:
model.eval()
correct=0
total=0
total_train_loss=0
with torch.no_grad():
    for x,y in train_loader:
        x=x.to(device)
        y=y.to(device)
        logits=model(x)
        loss1=loss(logits,y)
        predict=torch.argmax(logits,dim=1)
        correct+=(predict==y).sum().item()
        total+=y.size(0)
        total_train_loss+=loss1.item()
accuracy = correct / total
average_train_loss = total_train_loss / len(train_loader)

print("Train accuracy:", accuracy)
print("Train loss:", average_train_loss)

Train accuracy: 0.9163833333333333
Train loss: 0.21813080076207506


In [11]:
model.eval()

correct = 0
total = 0
total_test_loss = 0

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        loss1 = loss(logits, y)

        predict = torch.argmax(logits, dim=1)

        correct += (predict == y).sum().item()
        total += y.size(0)

        total_test_loss += loss1.item()

accuracy = correct / total
average_test_loss = total_test_loss / len(test_loader)

print("Test accuracy:", accuracy)
print("Test loss:", average_test_loss)

Test accuracy: 0.8861
Test loss: 0.3208161544054747
